# IT2226 — Étape 4 : EfficientNet-B0 comparé à ResNet-50

**Auteur :** _ton nom_  |  **Modèle entraîné ici :** EfficientNet-B0  |  **Référence :** ResNet-50 du notebook PR1

## Pourquoi ce notebook
`requirements.md` (R2) demande de choisir le modèle final **en comparant au moins deux candidats sur la validation**. ResNet-50 a été entraîné dans `PR1_notebook_ResNet50.ipynb`. Ici on entraîne EfficientNet-B0 avec **exactement le même protocole** (même manifest, mêmes augmentations, mêmes poids de classes, mêmes deux phases, même grille de learning rates, même seed). Seule l'architecture change.

## Le piège évité : le test a déjà été ouvert pour ResNet-50
Si on gardait « celui qui a le meilleur score de test », le test servirait à **choisir** et ne mesurerait plus rien d'impartial. La règle de choix est donc **fixée avant l'entraînement** (section 2), appliquée **sur la validation** (section 6), et **enregistrée sur Drive avant d'ouvrir le test** (section 7). Le test ne sert qu'au bilan et ne change pas le choix.

## Avant de commencer
Il faut sur ton Drive, avec le même compte Google :
- `MyDrive/dentalguide/eda_v2/` : le manifest du notebook 01 ;
- `MyDrive/dentalguide/pr1/` : les résultats du PR1 (`run_info.json`, `resnet50_dental.pt`, `training_history.csv`).

## Comment l'utiliser
1. Colab : **Exécution > Modifier le type d'exécution > GPU (T4)**.
2. Exécute les cellules **dans l'ordre**. Ne modifie **pas** la règle de la section 2 après avoir vu des résultats.
3. Aux endroits marqués ✍️, **écris toi-même** l'interprétation.

> ⚠️ Ce code a été généré avec l'aide de Claude. Tu dois pouvoir expliquer chaque cellule sans notes. Note tes vérifications dans ton AI Use Log.

## 1. Mise en place
Paramètres identiques au PR1, sauf `MODEL_NAME` et le dossier de sortie.

In [ ]:
!pip -q install kagglehub

In [ ]:
import os, random, hashlib, time, json, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from google.colab import drive

# ---------- Paramètres (identiques au PR1, sauf MODEL_NAME et DRIVE_OUT) ----------
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
MODEL_NAME = "efficientnet_b0"   # modèle entraîné dans ce notebook
REF_NAME = "resnet50"            # modèle de référence (PR1)

MANIFEST_DIR = Path("/content/drive/MyDrive/dentalguide/eda_v2")    # notebook 01
RESNET_DIR = Path("/content/drive/MyDrive/dentalguide/pr1")         # résultats du PR1
DRIVE_OUT = Path("/content/drive/MyDrive/dentalguide/effnet_b0")    # résultats de ce notebook
OUT_DIR = Path("outputs_effnet")
OUT_DIR.mkdir(exist_ok=True)

drive.mount("/content/drive")
assert (MANIFEST_DIR / "data_manifest.csv").exists(), "Manifest introuvable : lance d'abord 01_EDA_et_manifest_v2"
for f in ["run_info.json", f"{REF_NAME}_dental.pt", "training_history.csv"]:
    assert (RESNET_DIR / f).exists(), f"{f} introuvable dans {RESNET_DIR} : lance d'abord le notebook PR1"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
print("Appareil :", GPU_NAME, "| torch", torch.__version__, "| torchvision", torchvision.__version__)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


set_seed(SEED)

## 2. Règle de comparaison — fixée **avant** l'entraînement
Décision d'équipe du 18/11/2026 (option B) :

> EfficientNet-B0 **remplace** ResNet-50 comme modèle final **seulement si** :
> 1. son macro-F1 de validation dépasse celui de ResNet-50 (0,910) **d'au moins 0,02** ;
> 2. **et** son rappel de validation est **≥ 0,75 pour chaque classe**.
>
> Sinon, ResNet-50 reste le modèle final. Les deux modèles sont ensuite évalués **une fois** sur le test, pour le bilan, **sans changer le choix**.

**Pourquoi une marge de 0,02 ?** Chaque modèle n'est entraîné qu'avec une seule seed : un écart de quelques millièmes peut venir du hasard de l'entraînement. Cette marge est une **décision d'équipe**, pas un standard. Il faudra la justifier à l'oral.

Le macro-F1 de référence est relu dans `run_info.json` du PR1. La règle est enregistrée sur Drive. Si elle y existe déjà avec d'autres valeurs, la cellule s'arrête : on ne change pas la règle en cours de route.

In [ ]:
ref_info = json.loads((RESNET_DIR / "run_info.json").read_text())
assert ref_info["model"] == REF_NAME

RULE = {
    "fixed_on": "2026-11-18",
    "reference_model": REF_NAME,
    "reference_val_macro_f1": ref_info["best_val_macro_f1"],   # 0.9101 au PR1
    "margin": 0.02,             # gain minimal exigé sur le macro-F1 de validation
    "min_val_recall": 0.75,     # rappel de validation minimal, pour chaque classe
    "test_used_for_choice": False,
}

rule_file = DRIVE_OUT / "comparison_rule.json"
if rule_file.exists():
    previous = json.loads(rule_file.read_text())
    assert previous == RULE, f"Une autre règle est déjà enregistrée sur Drive : {previous}. On ne la change pas."
else:
    rule_file.write_text(json.dumps(RULE, indent=2))
shutil.copy(rule_file, OUT_DIR / "comparison_rule.json")

THRESHOLD = RULE["reference_val_macro_f1"] + RULE["margin"]
print(f"Règle enregistrée : EfficientNet-B0 devient le modèle final si macro-F1 val >= {THRESHOLD:.4f} "
      f"et rappel val >= {RULE['min_val_recall']} pour chaque classe. Sinon : {REF_NAME}.")

## 3. Les données (même manifest que le PR1)
L'EDA a déjà été faite dans les notebooks 01 et PR1 : on ne la refait pas. On télécharge le dataset, on relit le manifest, puis on **vérifie** que les fichiers et le découpage sont exactement ceux du PR1. Sinon, la comparaison ne serait pas équitable.

In [ ]:
import kagglehub

try:
    DATA_ROOT = Path(kagglehub.dataset_download("jhirshaikh/oral-image-dataset"))
except Exception as e:
    print("Téléchargement impossible :", e)
    DATA_ROOT = Path("/content/oral-image-dataset")   # <-- à adapter si besoin
print("Dataset dans :", DATA_ROOT)

In [ ]:
manifest = pd.read_csv(MANIFEST_DIR / "data_manifest.csv")
classes = json.loads((MANIFEST_DIR / "class_names.json").read_text())
assert classes == ref_info["classes"], "Ordre des classes différent du PR1"
class_to_idx = {c: i for i, c in enumerate(classes)}

manifest["path"] = manifest.rel_path.map(lambda r: str(DATA_ROOT / r))
missing = [p for p in manifest.path if not Path(p).exists()]
assert not missing, f"{len(missing)} images du manifest introuvables, par exemple {missing[:3]}"
md5 = manifest.path.map(lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest())   # 1 à 2 minutes
assert (md5 == manifest.md5).all(), "Un fichier ne correspond plus au manifest"
assert (manifest.label.map(class_to_idx) == manifest.y).all()

df = manifest
train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)

# Même découpage que celui du PR1 (2 827 / 484 / 399 images)
EXPECTED = {"train": 2827, "val": 484, "test": 399}
assert {"train": len(train_df), "val": len(val_df), "test": len(test_df)} == EXPECTED, "Découpage différent du PR1"
assert df.groupby("group_id").split.nunique().max() == 1, "Un groupe de copies est coupé entre deux jeux !"
print("Même manifest et même découpage que le PR1 :", EXPECTED)

## 4. Prétraitement, chargeurs et poids de classes (identiques au PR1)
On garde **exactement** le prétraitement du PR1 pour les deux modèles, pour que seule l'architecture diffère.

*Remarque :* la documentation de torchvision propose pour EfficientNet-B0 un prétraitement légèrement différent (méthode de redimensionnement). C'est à vérifier dans la documentation si un jury pose la question. Nous avons choisi le prétraitement commun, par équité.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet, comme au PR1
assert MEAN == ref_info["mean"] and STD == ref_info["std"] and IMG_SIZE == ref_info["img_size"]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class OralDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame.path.tolist()
        self.labels = frame.y.tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]


pin = DEVICE.type == "cuda"
train_loader = DataLoader(OralDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader = DataLoader(OralDataset(val_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(OralDataset(test_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=pin)

train_counts = np.bincount(train_df.y, minlength=len(classes))
class_weights = torch.tensor(train_counts.sum() / (len(classes) * train_counts), dtype=torch.float32)
print(pd.DataFrame({"classe": classes, "images train": train_counts, "poids": class_weights.numpy().round(2)}).to_string(index=False))

## 5. Le modèle et l'entraînement (même code que le PR1)
**EfficientNet-B0** est une autre famille de réseaux : au lieu d'empiler des blocs résiduels comme ResNet, il utilise des convolutions « séparables » plus légères et a été dimensionné pour obtenir une bonne précision avec peu de paramètres. Sa dernière couche (`classifier`) est remplacée par la **même tête** que ResNet-50 : `Dropout(0.3)` puis une couche linéaire à 6 sorties.

Les deux phases (tête seule, puis réseau entier avec un learning rate plus faible pour le corps) et la sélection sur le macro-F1 de validation sont **identiques** au PR1.

In [ ]:
def build_model(name, n_classes, pretrained=True):
    # Charge un modèle (pré-entraîné ImageNet si pretrained) et remplace sa dernière couche.
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, n_classes))
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, n_classes))
    else:
        raise ValueError("Modèle inconnu : " + name)
    return m.to(DEVICE)


def head_params(m):
    return (m.fc if hasattr(m, "fc") else m.classifier).parameters()


def set_backbone_trainable(m, trainable):
    head_ids = {id(p) for p in head_params(m)}
    for p in m.parameters():
        p.requires_grad = True if id(p) in head_ids else trainable


n_params = {}
for name in [REF_NAME, MODEL_NAME]:
    m = build_model(name, len(classes), pretrained=False)
    n_params[name] = sum(p.numel() for p in m.parameters())
    print(f"{name:16s}: {n_params[name]/1e6:.2f} M de paramètres (tête à 6 classes incluse)")
    del m

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss, preds, labels = 0.0, [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
                out = model(x)
                loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * x.size(0)
        preds += out.argmax(1).cpu().tolist()
        labels += y.cpu().tolist()
    n = len(labels)
    acc = float(np.mean(np.array(preds) == np.array(labels)))
    macro_f1 = f1_score(labels, preds, average="macro")
    return total_loss / n, acc, macro_f1


def run_experiment(tag, lr_head, lr_backbone, epochs_head, epochs_ft, verbose=True):
    # Identique au PR1 : phase 1 (tête seule) puis phase 2 (réseau entier).
    set_seed(SEED)
    model = build_model(MODEL_NAME, len(classes))
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    history, best_f1, best_state, baseline_f1 = [], -1.0, None, None

    def one_phase(phase, epochs, optimizer, scheduler):
        nonlocal best_f1, best_state
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, criterion, optimizer, scaler)
            va_loss, va_acc, va_f1 = run_epoch(model, val_loader, criterion)
            if scheduler:
                scheduler.step()
            history.append(dict(tag=tag, phase=phase, epoch=ep, train_loss=tr_loss, val_loss=va_loss,
                                train_f1=tr_f1, val_f1=va_f1, val_acc=va_acc))
            if verbose:
                print(f"[{tag}|{phase}] epoch {ep}/{epochs} train_loss {tr_loss:.3f} | "
                      f"val_loss {va_loss:.3f} val_acc {va_acc:.3f} val_macroF1 {va_f1:.3f}")
            if va_f1 > best_f1:
                best_f1 = va_f1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    set_backbone_trainable(model, False)
    opt1 = torch.optim.AdamW(head_params(model), lr=lr_head, weight_decay=1e-4)
    one_phase("phase1_tete", epochs_head, opt1, None)
    baseline_f1 = best_f1

    set_backbone_trainable(model, True)
    head_ids = {id(p) for p in head_params(model)}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    opt2 = torch.optim.AdamW([{"params": backbone, "lr": lr_backbone},
                              {"params": list(head_params(model)), "lr": lr_head / 10}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=epochs_ft)
    one_phase("phase2_finetuning", epochs_ft, opt2, sched)

    return dict(tag=tag, lr_head=lr_head, lr_backbone=lr_backbone, baseline_val_f1=baseline_f1,
                best_val_f1=best_f1, best_state=best_state, history=pd.DataFrame(history))

### 5.1 Essai rapide (1 epoch par phase)
On vérifie que tout tourne avant les vrais essais.

In [ ]:
t0 = time.time()
smoke = run_experiment("test_rapide", lr_head=1e-3, lr_backbone=1e-4, epochs_head=1, epochs_ft=1)
print(f"OK en {time.time() - t0:.0f} s | macro-F1 val : {smoke['best_val_f1']:.3f}")
del smoke

### 5.2 Même grille de learning rates que le PR1
Mêmes epochs (3 + 8) et mêmes trois learning rates pour le corps. On mesure aussi la **durée totale** de cette grille.

*Limite à connaître :* ces learning rates ont été choisis pour ResNet-50. Il est possible qu'EfficientNet-B0 préfère d'autres valeurs. On garde la même grille par équité et on note cette limite dans le bilan.

In [ ]:
EPOCHS_HEAD = 3
EPOCHS_FT = 8
LR_HEAD = 1e-3
LR_BACKBONE_CANDIDATES = [3e-5, 1e-4, 3e-4]

t_grid = time.time()
experiments = []
for lr_bb in LR_BACKBONE_CANDIDATES:
    print(f"\n===== {MODEL_NAME} | lr_backbone = {lr_bb} =====")
    experiments.append(run_experiment(f"lr_bb={lr_bb:g}", LR_HEAD, lr_bb, EPOCHS_HEAD, EPOCHS_FT))
GRID_MINUTES = (time.time() - t_grid) / 60
print(f"\nDurée de la grille (3 essais) : {GRID_MINUTES:.1f} min sur {GPU_NAME}")

results = pd.DataFrame([{k: v for k, v in e.items() if k not in ("best_state", "history")} for e in experiments])
results = results.sort_values("best_val_f1", ascending=False).reset_index(drop=True)
results.to_csv(OUT_DIR / "hyperparameter_results.csv", index=False)
results

In [ ]:
best = max(experiments, key=lambda e: e["best_val_f1"])
print(f"Meilleur essai EfficientNet-B0 (VALIDATION) : {best['tag']} | macro-F1 val {best['best_val_f1']:.4f} "
      f"(tête seule : {best['baseline_val_f1']:.4f})")

# Courbe de validation des deux modèles, chacun avec son meilleur learning rate
h_eff = best["history"].reset_index(drop=True)
h_ref = pd.read_csv(RESNET_DIR / "training_history.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h_eff.train_loss, label="train"); ax[0].plot(h_eff.val_loss, label="validation")
ax[0].axvline(EPOCHS_HEAD - 0.5, color="gray", ls="--"); ax[0].set_title("EfficientNet-B0 : perte"); ax[0].legend()
ax[0].set_xlabel("epoch")
ax[1].plot(h_ref.val_f1.values, label=f"ResNet-50 ({ref_info['best_experiment']})")
ax[1].plot(h_eff.val_f1, label=f"EfficientNet-B0 ({best['tag']})")
ax[1].axvline(EPOCHS_HEAD - 0.5, color="gray", ls="--"); ax[1].set_title("Macro-F1 de validation"); ax[1].legend()
ax[1].set_xlabel("epoch")
fig.suptitle("Pointillés : passage de la phase 1 (tête seule) à la phase 2 (fine-tuning)")
fig.tight_layout()
fig.savefig(OUT_DIR / "training_curves_comparison.png", dpi=150)
plt.show()

✍️ **Interprétation de l'entraînement (à écrire toi-même)** :
- Quel learning rate est le meilleur pour EfficientNet-B0 ? Est-ce le même que pour ResNet-50 ?
- Le fine-tuning apporte-t-il autant qu'avec ResNet-50 (comparer « tête seule » et « meilleur ») ?
- Les courbes montrent-elles du surapprentissage ?

_(Ton texte ici.)_

## 6. Application de la règle, **sur la validation uniquement**
- **Macro-F1** : on compare la valeur retenue lors de la sélection, calculée de la même façon que le 0,9101 de ResNet-50 au PR1.
- **Rappels** : on réévalue proprement le meilleur EfficientNet-B0 sur la validation.
- On recharge aussi le ResNet-50 du PR1 et on l'évalue sur la validation, pour le tableau final. Son macro-F1 doit être très proche de 0,910 ; une petite différence est possible à cause du calcul en précision mixte.

La décision est **enregistrée sur Drive avant** la section 7.

In [ ]:
def predict(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            ps += model(x.to(DEVICE)).argmax(1).cpu().tolist()
            ys += y.tolist()
    return np.array(ys), np.array(ps)


eff_model = build_model(MODEL_NAME, len(classes), pretrained=False)
eff_model.load_state_dict(best["best_state"])
ref_model = build_model(REF_NAME, len(classes), pretrained=False)
ref_model.load_state_dict(torch.load(RESNET_DIR / f"{REF_NAME}_dental.pt", map_location=DEVICE))

yv, pv_eff = predict(eff_model, val_loader)
_, pv_ref = predict(ref_model, val_loader)
print("=== VALIDATION — EfficientNet-B0 ===")
print(classification_report(yv, pv_eff, target_names=classes, digits=3))
print("=== VALIDATION — ResNet-50 (rechargé depuis le PR1) ===")
print(classification_report(yv, pv_ref, target_names=classes, digits=3))

eff_val_f1 = float(best["best_val_f1"])
eff_val_recall = recall_score(yv, pv_eff, average=None, labels=range(len(classes)))
cond_f1 = eff_val_f1 >= THRESHOLD
low = [classes[i] for i, r in enumerate(eff_val_recall) if r < RULE["min_val_recall"]]
cond_recall = not low
FINAL_MODEL = MODEL_NAME if (cond_f1 and cond_recall) else REF_NAME

decision = {
    "decided_on_validation_only": True,
    "efficientnet_val_macro_f1": round(eff_val_f1, 4),
    "threshold": round(THRESHOLD, 4),
    "condition_macro_f1": bool(cond_f1),
    "classes_below_min_recall": low,
    "condition_recall": bool(cond_recall),
    "final_model": FINAL_MODEL,
}
decision_file = DRIVE_OUT / "decision.json"
if decision_file.exists():
    print("Décision déjà enregistrée lors d'une exécution précédente :", json.loads(decision_file.read_text()))
decision_file.write_text(json.dumps(decision, indent=2))
shutil.copy(decision_file, OUT_DIR / "decision.json")

print(f"Macro-F1 val EfficientNet-B0 {eff_val_f1:.4f} >= {THRESHOLD:.4f} : {'OUI' if cond_f1 else 'NON'}")
print("Classes sous le rappel minimal :", low if low else "aucune")
print(f"\n==> MODÈLE FINAL (décidé sur la validation) : {FINAL_MODEL}")

## 7. Test : une seule fois, pour le bilan
Le choix est fait et enregistré. On évalue maintenant les deux modèles sur le test. **Ces résultats ne changent pas le choix.**

Pour ResNet-50, on doit retrouver les résultats du PR1 (macro-F1 0,904, 29 erreurs) : c'est aussi un contrôle de reproductibilité.

On relit aussi les critères de réussite du PR1 (macro-F1 ≥ 0,80, rappel ≥ 0,75 par classe pathologique) pour situer EfficientNet-B0.

In [ ]:
assert (DRIVE_OUT / "decision.json").exists(), "La décision doit être enregistrée avant d'ouvrir le test (section 6)."

yt, pt_eff = predict(eff_model, test_loader)
_, pt_ref = predict(ref_model, test_loader)
crit = ref_info["success_criteria"]
pathological = [i for i, c in enumerate(classes) if c != "Normal"]

for name, pt in [(REF_NAME, pt_ref), (MODEL_NAME, pt_eff)]:
    f1 = f1_score(yt, pt, average="macro")
    rec = recall_score(yt, pt, average=None, labels=range(len(classes)))
    low_t = [classes[i] for i in pathological if rec[i] < crit["min_recall"]]
    print(f"=== TEST — {name} | accuracy {(yt == pt).mean():.3f} | macro-F1 {f1:.3f} | "
          f"erreurs {(yt != pt).sum()}/{len(yt)} ===")
    print(classification_report(yt, pt, target_names=classes, digits=3))
    print(f"Critères PR1 : macro-F1 >= {crit['min_macro_f1']} : {'OUI' if f1 >= crit['min_macro_f1'] else 'NON'} | "
          f"classes pathologiques sous {crit['min_recall']} : {low_t if low_t else 'aucune'}\n")

(OUT_DIR / "classification_report_test.txt").write_text(
    classification_report(yt, pt_eff, target_names=classes, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (name, pt) in zip(axes, [(REF_NAME, pt_ref), (MODEL_NAME, pt_eff)]):
    cm = confusion_matrix(yt, pt, labels=range(len(classes)))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel("Classe prédite"); ax.set_ylabel("Vraie classe"); ax.set_title(f"{name} (test)")
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.tight_layout()
fig.savefig(OUT_DIR / "confusion_matrices_comparison.png", dpi=150)
plt.show()

## 8. Tableau comparatif
On ajoute deux mesures utiles pour l'application locale (étape 6) :
- **Taille du modèle** (nombre de paramètres et mégaoctets des poids) ;
- **Temps d'inférence** : sur le GPU de Colab (lot de 32 images) et sur le **CPU de Colab** (1 image à la fois, plus proche de l'usage dans l'application). Ce ne sont **pas** les temps de ta machine : seulement un ordre de grandeur relatif entre les deux modèles. Le chargement des images n'est pas compté.

La durée d'entraînement de ResNet-50 n'a pas été mesurée au PR1 : elle apparaît comme « non mesurée ».

In [ ]:
def ms_gpu_batch(model, loader, n_batches=10):
    # millisecondes par image, lots de BATCH_SIZE, sur le GPU (chargement des images exclu)
    if DEVICE.type != "cuda":
        return float("nan")
    model.eval(); total_t, total_n = 0.0, 0
    with torch.no_grad():
        for i, (x, _) in enumerate(loader):
            x = x.to(DEVICE)
            if i == 0:
                model(x); torch.cuda.synchronize(); continue   # échauffement
            t = time.perf_counter(); model(x); torch.cuda.synchronize()
            total_t += time.perf_counter() - t; total_n += x.size(0)
            if i >= n_batches:
                break
    return 1000 * total_t / total_n


def ms_cpu_single(model, n_images=20):
    # millisecondes par image, une image à la fois, sur le CPU de Colab
    m = model.to("cpu").eval()
    x = torch.stack([eval_tf(Image.open(p).convert("RGB")) for p in val_df.path[:n_images]])
    with torch.no_grad():
        m(x[:1])   # échauffement
        t = time.perf_counter()
        for i in range(n_images):
            m(x[i:i + 1])
    model.to(DEVICE)
    return 1000 * (time.perf_counter() - t) / n_images


def size_mb(model):
    return sum(v.numel() * v.element_size() for v in model.state_dict().values()) / 1e6


def column(name, model, val_f1_sel, pv, pt, train_minutes):
    rv = recall_score(yv, pv, average=None, labels=range(len(classes)))
    rt = recall_score(yt, pt, average=None, labels=range(len(classes)))
    col = {
        "paramètres (M)": round(n_params[name] / 1e6, 2),
        "poids (Mo)": round(size_mb(model), 1),
        "durée grille LR (min)": train_minutes,
        "VAL macro-F1 (sélection)": round(val_f1_sel, 4),
    }
    col.update({f"VAL rappel {c}": round(float(r), 3) for c, r in zip(classes, rv)})
    col["TEST accuracy"] = round(float((yt == pt).mean()), 4)
    col["TEST macro-F1"] = round(float(f1_score(yt, pt, average="macro")), 4)
    col.update({f"TEST rappel {c}": round(float(r), 3) for c, r in zip(classes, rt)})
    col["TEST erreurs"] = int((yt != pt).sum())
    col["inférence GPU (ms/image, lot 32)"] = round(ms_gpu_batch(model, val_loader), 2)
    col["inférence CPU Colab (ms/image, 1 image)"] = round(ms_cpu_single(model), 1)
    return col


comparison = pd.DataFrame({
    "ResNet-50": column(REF_NAME, ref_model, ref_info["best_val_macro_f1"], pv_ref, pt_ref, "non mesurée"),
    "EfficientNet-B0": column(MODEL_NAME, eff_model, eff_val_f1, pv_eff, pt_eff, round(GRID_MINUTES, 1)),
})
comparison.to_csv(OUT_DIR / "comparison.csv")
print(f"Modèle final (règle appliquée sur la validation) : {FINAL_MODEL}")
comparison

✍️ **Bilan comparatif (à écrire toi-même)** :
- Que dit la règle, et pourquoi a-t-on fixé une marge avant d'entraîner ?
- Sur la validation, puis sur le test : l'écart entre les deux modèles est-il grand ou dans le bruit (une seule seed, 484 et 399 images) ?
- Les deux modèles font-ils les **mêmes** erreurs (comparer les matrices) ? Calculus et Caries restent-ils confondus ?
- Taille et vitesse : EfficientNet-B0 est-il plus intéressant pour une application locale, même à performance égale ? (Rappel : la règle fixée ne prend en compte que la performance de validation.)
- Limites : une seule seed, grille de learning rates choisie pour ResNet-50, temps mesurés sur Colab et pas sur notre machine.

_(Ton texte ici.)_

## 9. Sauvegarde
Tout est copié dans `MyDrive/dentalguide/effnet_b0/`. Les poids (`.pt`) restent sur Drive, comme ceux de ResNet-50 ; les autres fichiers sont à importer dans `data/effnet_b0/` du dépôt.

In [ ]:
torch.save(best["best_state"], OUT_DIR / f"{MODEL_NAME}_dental.pt")
(OUT_DIR / "classes.json").write_text(json.dumps(classes))
best["history"].to_csv(OUT_DIR / "training_history.csv", index=False)

run_info = {
    "model": MODEL_NAME,
    "classes": classes,
    "best_experiment": best["tag"],
    "best_val_macro_f1": round(eff_val_f1, 4),
    "test_accuracy": round(float((yt == pt_eff).mean()), 4),
    "test_macro_f1": round(float(f1_score(yt, pt_eff, average="macro")), 4),
    "comparison_rule": RULE,
    "final_model_by_rule": FINAL_MODEL,
    "split_source": "data_manifest.csv (01_EDA_et_manifest_v2)",
    "img_size": IMG_SIZE, "mean": MEAN, "std": STD,
    "torch": torch.__version__, "torchvision": torchvision.__version__, "seed": SEED,
    "gpu": GPU_NAME, "grid_minutes": round(GRID_MINUTES, 1),
}
(OUT_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2))

shutil.copytree(OUT_DIR, DRIVE_OUT, dirs_exist_ok=True)
print("Fichiers sauvegardés sur Drive :", sorted(p.name for p in DRIVE_OUT.iterdir()))

## 10. Checklist
- [ ] La règle de la section 2 n'a pas été modifiée après avoir vu des résultats
- [ ] Notebook exécuté **de haut en bas** sans erreur, toutes les sorties visibles
- [ ] ResNet-50 rechargé redonne bien les résultats de test du PR1
- [ ] Les 2 cellules ✍️ sont remplies **avec tes mots**
- [ ] Fichiers de `MyDrive/dentalguide/effnet_b0/` (sauf le `.pt`) importés dans `data/effnet_b0/`
- [ ] AI Use Log mis à jour